# Week 7, Day 1: Quantize, attach, count

**Lesson:** [the Colab notebook](https://colab.research.google.com/drive/1oE6jCl8yUxsT1V4N43BD_0Wx5u0A9AQe) linked from [week7/day1.ipynb](../../../week7/day1.ipynb), *QLoRA*. **About 25 minutes.**

**What you practice here**

- Loading a model at full precision, in 8 bits and in 4 bits with `BitsAndBytesConfig` and `AutoModelForCausalLM.from_pretrained`, and measuring each load with `get_memory_footprint`.
- Putting a trained LoRA adapter on the 4-bit base with `PeftModel.from_pretrained`, and measuring what it adds.
- Counting an adapter's weights from the module shapes: r × (d_in + d_out) for each targeted module, in every layer, at 4 bytes a weight.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week7/day1", globals())
else:  # in Colab, or outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here, as in Colab.")

## The task

The lesson loads Llama 3.2 3B three times, at 32, 8 and 4 bits, and watches its memory footprint shrink. Then it
puts a trained LoRA adapter on the 4-bit model, and counts the weights that LoRA adds. Do the same with a bigger
model in mind: would Llama 3.1 8B, with a LoRA adapter that prices wrestling merchandise, fit on Colab's free T4?

transformers and peft are not in the checks' Python, so `load_base` and `attach_adapter` take the classes they use
as parameters (`AutoModelForCausalLM` and `BitsAndBytesConfig`, then `PeftModel`), and call them as the lesson does.
The checks hand them stand-ins: a loader whose models report their memory footprint, a strict
`BitsAndBytesConfig`, and an adapter loader that adds the adapter's bytes to the model it is given.

## What each call should give

```python
quant_settings(8)
# {'load_in_8bit': True}
quant_settings(4)
# the lesson's four 4-bit settings, as a dict: 4 bits, double quantization, the NF4 type, compute in "bfloat16"

model = load_base(AutoModelForCausalLM, BitsAndBytesConfig, bits=4, name="meta-llama/Llama-3.1-8B")
# loads Llama 3.1 8B once, placed on the GPU, with a BitsAndBytesConfig made from quant_settings(4)

tuned, added = attach_adapter(PeftModel, model, "someone/wwe-merch-lora")
# tuned: the model with the adapter on it; added: the megabytes the adapter put on it, such as 54.525952

LLAMA_8B = {"q_proj": (4096, 4096), "k_proj": (4096, 1024), "v_proj": (4096, 1024), "o_proj": (4096, 4096),
            "gate_proj": (4096, 14336), "up_proj": (4096, 14336), "down_proj": (14336, 4096)}
lora_size(LLAMA_8B, ["q_proj", "k_proj", "v_proj", "o_proj"], 16, 32)
# (13631488, 54.525952)
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.

### 1 · `quant_settings(bits)`

Returns the settings of the lesson's quantization config, as a dict of keyword arguments for `BitsAndBytesConfig`.
For `bits=8`, there is one setting: load in 8 bits. For `bits=4`, there are the lesson's four: load in 4 bits, use
double quantization, use the NF4 quantization type (`"nf4"`), and compute in bfloat16, given as the string
`"bfloat16"`. The class turns that name into `torch.bfloat16` itself, so this part needs no torch. Return exactly
these settings, under the names `BitsAndBytesConfig` takes. For any other `bits`, raise `ValueError`.

<details><summary>Hint</summary>

```python
{"load_in_8bit": True}
{"load_in_4bit": True, "bnb_4bit_use_double_quant": True, "bnb_4bit_compute_dtype": "bfloat16",
 "bnb_4bit_quant_type": "nf4"}
```

</details>

In [ ]:
def quant_settings(bits):
    # Write quant_settings here, as described above.
    raise NotImplementedError("quant_settings is not written yet")

In [ ]:
check("quant_settings")

### 2 · `load_base(auto_model, bnb_config, bits=None, name="meta-llama/Llama-3.2-3B")`

Loads the base model at full precision, or quantized to 8 or 4 bits, and returns it. `auto_model` is
`AutoModelForCausalLM` and `bnb_config` is `BitsAndBytesConfig`. Load the model `name` once, with the loader that
`auto_model` has, and let the loader place it on the GPU, as the lesson's `device_map="auto"` does. Pass every
setting by name. With `bits=None`, pass no quantization config, as the lesson's first load does. With 8 or 4, make
a config with `bnb_config` from `quant_settings(bits)`, and pass it as the quantization config.

<details><summary>Hint</summary>

```python
auto_model.from_pretrained(name, device_map="auto")
auto_model.from_pretrained(name, quantization_config=bnb_config(**quant_settings(bits)), device_map="auto")
```

</details>

In [ ]:
def load_base(auto_model, bnb_config, bits=None, name="meta-llama/Llama-3.2-3B"):
    # Write load_base here, as described above.
    raise NotImplementedError("load_base is not written yet")

In [ ]:
check("load_base")

### 3 · `attach_adapter(peft_model, base_model, adapter_id)`

Puts a trained LoRA adapter on the model, and returns the adapted model with the megabytes the adapter added.
`peft_model` is peft's `PeftModel`. Read the base model's memory footprint, which is in bytes. Then load the adapter
`adapter_id` onto `base_model` with the loader that `PeftModel` has, and read the footprint of the model that it
returns. Return `(fine_tuned_model, added_mb)`, where `added_mb` is the difference between the two footprints,
divided by 1e6.

Read the first footprint before you attach. The loader puts the LoRA layers into the base model itself, so once the
adapter is on, the base model's footprint has grown too.

<details><summary>Hint</summary>

```python
before = base_model.get_memory_footprint()
fine_tuned_model = peft_model.from_pretrained(base_model, adapter_id)
added_mb = (fine_tuned_model.get_memory_footprint() - before) / 1e6
```

</details>

In [ ]:
def attach_adapter(peft_model, base_model, adapter_id):
    # Write attach_adapter here, as described above.
    raise NotImplementedError("attach_adapter is not written yet")

In [ ]:
check("attach_adapter")

### 4 · `lora_size(shapes, targets, r, layers)`

Counts the weights a LoRA adapter adds, and their size in megabytes. `shapes` maps each module name to its
`(d_in, d_out)`, such as `{"q_proj": (4096, 4096), "k_proj": (4096, 1024)}`, and `targets` lists the modules that
LoRA adapts, like `target_modules`. Each targeted module gets two matrices, lora_A with r × d_in weights and lora_B
with d_out × r, and every layer has its own pair. Return `(params, megabytes)`: the weights of the targeted modules
in one layer, times `layers`, and their size at 4 bytes a weight, divided by 1e6, as the lesson counts. Leave out
the modules that are not in `targets`.

<details><summary>Hint</summary>

```python
params = layers * sum(r * (shapes[name][0] + shapes[name][1]) for name in targets)
return params, params * 4 / 1e6
```

</details>

In [ ]:
def lora_size(shapes, targets, r, layers):
    # Write lora_size here, as described above.
    raise NotImplementedError("lora_size is not written yet")

In [ ]:
check("lora_size")

## Check everything

The cell below runs all 8 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 7, Day 1 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. This needs a GPU, so it runs in Google Colab: push this notebook, then [open it in Colab](https://colab.research.google.com/github/starlest/llm_engineering/blob/main/plan/exercises/week7/day1.ipynb), choose a T4 runtime, and run the cells from the top. In Colab the checks are off, so run them here in your codespace. Write the steps in the 2 cells at the end.

Colab works on its own copy of the notebook. To keep the Try code you write there, use **File > Save a copy in GitHub** (to this notebook's path in your fork), then run `git pull` in your codespace before you commit there again. Your parts are already in your fork, which is what the tracker reads.

Colab's free T4 is enough. Accept the Llama 3.2 license on its Hugging Face page first, and add your token to
Colab's secrets as `HF_TOKEN`. Loading at full precision fills most of the T4's memory, so restart the session
between loads, as the lesson does, and run the cells from the top again.

1. Keep the `!pip install` line in the first Try cell, and log in to Hugging Face with your `HF_TOKEN` secret.
2. Load Llama 3.2 3B with `load_base` at `bits=None`, then 8, then 4, restarting the session between loads
   (Runtime > Restart session, then run the cells from the top), and print the footprint in GB each time.
3. With the 4-bit model, attach the lesson's lite adapter, `ed-donner/price-2025-11-30_15.10.55-lite`, with
   `attach_adapter`. Print the megabytes it added, and print the model to find `lora_A` and `lora_B`.
4. Compare those megabytes with `lora_size` for Llama 3.2 3B's four attention modules (`q_proj` and `o_proj` are
   3072 by 3072; `k_proj` and `v_proj` are 3072 by 1024), at r = 32 over 28 layers. They should be about 73.4 MB.
5. Optional: load `meta-llama/Llama-3.1-8B` at 4 bits, and see that it fits on the T4. Accept its license on its
   model page first.

<details><summary>Stuck? One way to write it</summary>

```python
!pip install -q --upgrade bitsandbytes
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

login(userdata.get("HF_TOKEN"), add_to_git_credential=True)
BITS = 4  # run with None, then 8, then 4: Runtime > Restart session between them
model = load_base(AutoModelForCausalLM, BitsAndBytesConfig, bits=BITS)
print(f"{BITS or 32} bits: {model.get_memory_footprint() / 1e9:,.1f} GB")
```

```python
from peft import PeftModel

tuned, added = attach_adapter(PeftModel, model, "ed-donner/price-2025-11-30_15.10.55-lite")
print(tuned)  # lora_A and lora_B sit inside q_proj, k_proj, v_proj and o_proj
attention = {"q_proj": (3072, 3072), "k_proj": (3072, 1024), "v_proj": (3072, 1024), "o_proj": (3072, 3072)}
params, mb = lora_size(attention, list(attention), 32, 28)
print(f"The adapter added {added:,.1f} MB; your count: {params:,} weights, {mb:,.1f} MB")

# Optional, after a restart: Llama 3.1 8B in 4 bits
# big = load_base(AutoModelForCausalLM, BitsAndBytesConfig, bits=4, name="meta-llama/Llama-3.1-8B")
# print(f"Llama 3.1 8B in 4 bits: {big.get_memory_footprint() / 1e9:,.1f} GB")
```

</details>

In [ ]:
!pip install -q --upgrade bitsandbytes
# Steps 1 and 2: log in, then load Llama 3.2 3B at bits None, 8 and 4, restarting the session between loads

In [ ]:
# Steps 3 and 4: with the 4-bit model, attach the lesson's lite adapter and compare its size with lora_size